<a href="https://colab.research.google.com/github/ReNZzyi/IT-and-Programming/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 10. Использование анонимных функций

ФИО студента: **Петров Данила Максимович**

Группа студента: **ИНБ-б-о-26-2**

Вариант: **14**

## Задание

Продемонстрируйте использование лямбда-выражений для решения задач фильтрации, сортировки и отображения с использованием наборов данных из практического занятия №7.

**Вариант 14** - файл `p7_data_14.csv`: данные о товарах (ID, наименование, категория (4 шт.), количество, цена, признак просрочки, признак наличия скидки).

С помощью лямбда-выражений также решим задачи варианта 14 из практической работы №7:
1. Средняя цена товаров для просроченных и непросроченных товаров.
2. Суммарный объём в валюте товаров без категории.
3. Самый дорогой товар в каждой категории.
4. Количество наименований товаров без скидки.

## Решение

Файл `p7_data_14.csv` расположен в репозитории. Выполним копирование файла в локальное хранилище Google Colab, используя следующую команду:

In [ ]:
!wget https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract7/p7_data_14.csv

Можно скопировать и поместить файл с данными на виртуальную машину Google Colab любым удобным способом

Посмотрим на первые строки файла:

In [2]:
with open('p7_data_14.csv', encoding='utf-8') as f:
    for _ in range(7):
        print(f.readline().rstrip())

ID,Name,Category,Count,Price,OutOfDate,WithDiscount
1,"Fish - Atlantic Salmon, Cold",D,16,$60.49,true,true
2,Sauce - Cranberry,B,26,$19.52,true,true
3,Pan Grease,D,22,$20.87,,true
4,Coffee Guatemala Dark,A,10,$31.81,,true
5,Wine - Carmenere Casillero Del,,22,$75.15,false,
6,"Olives - Black, Pitted",,5,$95.04,,


Особенности набора данных:
* часть наименований содержит запятую и поэтому заключена в кавычки (`"Fish - Atlantic Salmon, Cold"`). Простое `line.split(',')`, как в учебном примере, разобьёт такую строку на лишние поля, поэтому используем модуль `csv`, который корректно обрабатывает кавычки;
* цена записана со знаком `$`;
* в полях `Category`, `OutOfDate`, `WithDiscount` встречаются пустые значения. Пустую категорию считаем «без категории», а пустой признак - «нет данных» (`None`), чтобы не путать его с `False`.

Проверим, сколько строк сломалось бы при наивном разбиении:

In [3]:
with open('p7_data_14.csv', encoding='utf-8') as f:
    f.readline()
    broken = list(filter(lambda line: len(line.strip().split(',')) != 7, f))
print('Строк, которые split(",") разобьёт неправильно:', len(broken))

Строк, которые split(",") разобьёт неправильно: 211


Реализуем чтение строк и их преобразование в словари:

In [4]:
import csv


def parse_flag(value):
    return {'true': True, 'false': False}.get(value)


products = []
with open('p7_data_14.csv', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f):
        product = {
            'ID': int(row['ID']),
            'Name': row['Name'],
            'Category': row['Category'] or None,
            'Count': int(row['Count']),
            'Price': float(row['Price'].lstrip('$')),
            'OutOfDate': parse_flag(row['OutOfDate']),
            'WithDiscount': parse_flag(row['WithDiscount']),
        }
        products.append(product)
print(len(products))

1000


Посмотрим на формат получившегося элемента списка:

In [5]:
print(products[0])
print(products[5])

{'ID': 1, 'Name': 'Fish - Atlantic Salmon, Cold', 'Category': 'D', 'Count': 16, 'Price': 60.49, 'OutOfDate': True, 'WithDiscount': True}
{'ID': 6, 'Name': 'Olives - Black, Pitted', 'Category': None, 'Count': 5, 'Price': 95.04, 'OutOfDate': None, 'WithDiscount': None}


Вспомогательная функция для табличного вывода товаров (чтобы не повторять один и тот же код печати):

In [6]:
def print_products(items, limit=10):
    print(f"{'ID':>4} | {'Наименование':<32} | Кат. | Кол. |   Цена | Просрочен | Скидка")
    print('-' * 85)
    for p in items[:limit]:
        print(f"{p['ID']:>4} | {p['Name'][:32]:<32} | {p['Category'] or '-':^4} | "
              f"{p['Count']:>4} | {p['Price']:>6.2f} | {str(p['OutOfDate']):^9} | "
              f"{str(p['WithDiscount']):^6}")

### Сортировка

Решим задачу: найти 10 самых дорогих товаров. Ключ сортировки задаётся лямбда-выражением, `reverse=True` - по убыванию. Функция `sorted()` возвращает новый список и не меняет исходный.

In [7]:
by_price = sorted(products, key=lambda p: p['Price'], reverse=True)
print_products(by_price)

  ID | Наименование                     | Кат. | Кол. |   Цена | Просрочен | Скидка
-------------------------------------------------------------------------------------
 563 | Pork Loin Bine - In Frenched     |  C   |   50 |  99.86 |   False   |  None 
 479 | Wine - White, Concha Y Toro      |  -   |    1 |  99.84 |   None    | False 
 664 | Country Roll                     |  B   |   18 |  99.76 |   True    |  None 
 257 | Chutney Sauce                    |  B   |    1 |  99.65 |   False   | False 
 155 | Juice - Grape, White             |  D   |    3 |  99.63 |   False   |  True 
 750 | Bouq All Italian - Primerba      |  A   |   26 |  99.63 |   False   |  True 
 675 | Bread - Wheat Baguette           |  C   |   31 |  99.58 |   False   | False 
 126 | Soup - Campbells, Butternut      |  -   |    9 |  99.55 |   False   |  True 
 995 | Muffin Mix - Corn Harvest        |  A   |   31 |  99.47 |   True    |  True 
 323 | Tea Peppermint                   |  -   |   47 |  99.46 |   True   

Сортировка по вычисляемому значению: стоимость партии товара на складе (количество × цена)

In [8]:
by_total = sorted(products, key=lambda p: p['Count'] * p['Price'], reverse=True)
for p in by_total[:5]:
    print(f"{p['Name']:<35} {p['Count']:>3} x {p['Price']:>6.2f} = {p['Count'] * p['Price']:>8.2f}")

Pork Loin Bine - In Frenched         50 x  99.86 =  4993.00
Absolut Citron                       50 x  95.08 =  4754.00
Cherries - Bing, Canned              50 x  93.76 =  4688.00
Tea Peppermint                       47 x  99.46 =  4674.62
Sherry - Dry                         47 x  97.00 =  4559.00


Сортировка по нескольким ключам: лямбда возвращает кортеж, а кортежи сравниваются поэлементно. Сначала по категории (товары без категории - в конце), внутри категории - по убыванию цены (знак минус у числа).

In [9]:
by_cat_price = sorted(products,
                      key=lambda p: (p['Category'] is None,
                                     p['Category'] or '',
                                     -p['Price']))
for cat in ['A', 'B', 'C', 'D', None]:
    group = list(filter(lambda p: p['Category'] == cat, by_cat_price))
    print(f"Категория {cat or 'нет'}:",
          ', '.join(f"{p['Name']} ({p['Price']})" for p in group[:3]))

Категория A: Bouq All Italian - Primerba (99.63), Muffin Mix - Corn Harvest (99.47), Sea Bass - Fillets (99.25)
Категория B: Country Roll (99.76), Chutney Sauce (99.65), Dill Weed - Dry (98.57)
Категория C: Pork Loin Bine - In Frenched (99.86), Bread - Wheat Baguette (99.58), Tea - Black Currant (98.78)
Категория D: Juice - Grape, White (99.63), Wine - Stoneliegh Sauvignon (98.42), Sobe - Berry Energy (97.54)
Категория нет: Wine - White, Concha Y Toro (99.84), Soup - Campbells, Butternut (99.55), Tea Peppermint (99.46)


### Фильтрация

Решим задачу: найти непросроченные товары со скидкой дешевле $15. Условие в лямбде - логическое выражение; `filter()` оставляет элементы, для которых оно истинно.

In [10]:
target = filter(lambda p: p['OutOfDate'] is False and
                          p['WithDiscount'] is True and
                          p['Price'] < 15,
                products)
target = list(target)
print('Найдено товаров:', len(target))
print_products(sorted(target, key=lambda p: p['Price']))

Найдено товаров: 24
  ID | Наименование                     | Кат. | Кол. |   Цена | Просрочен | Скидка
-------------------------------------------------------------------------------------
  19 | Soup Campbells Mexicali Tortilla |  B   |   45 |   5.32 |   False   |  True 
 725 | Sea Bass - Fillets               |  A   |   20 |   5.56 |   False   |  True 
 161 | Sponge Cake Mix - Chocolate      |  B   |   21 |   6.96 |   False   |  True 
 953 | Tomatoes - Roma                  |  C   |   48 |   7.01 |   False   |  True 
 749 | Sugar - Brown                    |  B   |   15 |   7.24 |   False   |  True 
 726 | Wine - Pinot Noir Latour         |  C   |    2 |   7.49 |   False   |  True 
 643 | Bread - Multigrain Oval          |  A   |    5 |   7.53 |   False   |  True 
 422 | Tart Shells - Savory, 3          |  B   |   27 |   7.82 |   False   |  True 
 137 | Lid - High Heat, Super Clear     |  C   |   40 |   8.56 |   False   |  True 
 360 | Mcgillicuddy Vanilla Schnap      |  -   |   45 

Фильтрация по тексту: все вина (наименование начинается с `Wine`) категории A

In [11]:
wines = list(filter(lambda p: p['Name'].startswith('Wine') and p['Category'] == 'A',
                    products))
print('Вин в категории A:', len(wines))
print_products(wines, limit=5)

Вин в категории A: 13
  ID | Наименование                     | Кат. | Кол. |   Цена | Просрочен | Скидка
-------------------------------------------------------------------------------------
   8 | Wine - Prem Select Charddonany   |  A   |    7 |  26.36 |   False   | False 
  71 | Wine - Chablis 2003 Champs       |  A   |   21 |  78.07 |   False   |  None 
 286 | Wine - Vouvray Cuvee Domaine     |  A   |   50 |  25.10 |   True    |  None 
 524 | Wine - Vineland Estate Semi - Dr |  A   |    6 |  90.90 |   None    | False 
 630 | Wine - Jaboulet Cotes Du Rhone   |  A   |   21 |  90.40 |   False   |  True 


### Отображение

Решим задачу: на основе имеющегося списка построить новый, содержащий только наименование и стоимость партии

In [12]:
target = map(lambda p: (p['Name'], round(p['Count'] * p['Price'], 2)), products)
target = list(target)
print(target[:3])

[('Fish - Atlantic Salmon, Cold', 967.84), ('Sauce - Cranberry', 507.52), ('Pan Grease', 459.14)]


Отображение с условным выражением внутри лямбды: цена с учётом скидки 10% (только для товаров, у которых скидка есть)

In [13]:
DISCOUNT = 0.10
with_final_price = list(map(
    lambda p: {'Name': p['Name'],
               'Price': p['Price'],
               'Final': round(p['Price'] * (1 - DISCOUNT), 2) if p['WithDiscount'] else p['Price']},
    products))
for item in with_final_price[:6]:
    print(item)

{'Name': 'Fish - Atlantic Salmon, Cold', 'Price': 60.49, 'Final': 54.44}
{'Name': 'Sauce - Cranberry', 'Price': 19.52, 'Final': 17.57}
{'Name': 'Pan Grease', 'Price': 20.87, 'Final': 18.78}
{'Name': 'Coffee Guatemala Dark', 'Price': 31.81, 'Final': 28.63}
{'Name': 'Wine - Carmenere Casillero Del', 'Price': 75.15, 'Final': 75.15}
{'Name': 'Olives - Black, Pitted', 'Price': 95.04, 'Final': 95.04}


Отображение, фильтрация и сортировка в одной цепочке: 5 самых дорогих (по стоимости партии) просроченных товаров - их списание обойдётся дороже всего

In [14]:
expired_losses = sorted(
    map(lambda p: (p['Name'], round(p['Count'] * p['Price'], 2)),
        filter(lambda p: p['OutOfDate'] is True, products)),
    key=lambda pair: pair[1],
    reverse=True)
for name, loss in expired_losses[:5]:
    print(f'{name:<35} {loss:>9.2f}')

Absolut Citron                        4754.00
Tea Peppermint                        4674.62
Sherry - Dry                          4559.00
Wine - Piper Heidsieck Brut           4488.03
Chevere Logs                          4458.89


## Индивидуальное задание (вариант 14)

**1. Средняя цена товаров для просроченных и непросроченных товаров.** Товары, у которых признак просрочки не указан, не относятся ни к одной из групп и в расчёт не входят.

In [15]:
def average(values):
    values = list(values)
    return sum(values) / len(values) if values else 0.0


expired = list(filter(lambda p: p['OutOfDate'] is True, products))
not_expired = list(filter(lambda p: p['OutOfDate'] is False, products))
unknown = list(filter(lambda p: p['OutOfDate'] is None, products))

print(f"Просроченные:   {len(expired):>3} шт., средняя цена ${average(map(lambda p: p['Price'], expired)):.2f}")
print(f"Непросроченные: {len(not_expired):>3} шт., средняя цена ${average(map(lambda p: p['Price'], not_expired)):.2f}")
print(f"Нет данных:     {len(unknown):>3} шт. (не учитываются)")

Просроченные:   421 шт., средняя цена $52.25
Непросроченные: 413 шт., средняя цена $49.74
Нет данных:     166 шт. (не учитываются)


**2. Суммарный объём в валюте товаров без категории** (сумма количество × цена)

In [16]:
no_category = list(filter(lambda p: p['Category'] is None, products))
total_no_category = sum(map(lambda p: p['Count'] * p['Price'], no_category))
print(f'Товаров без категории: {len(no_category)}')
print(f'Суммарный объём в валюте: ${total_no_category:,.2f}')

Товаров без категории: 208
Суммарный объём в валюте: $262,807.34


**3. Самый дорогой товар в каждой категории.** Функция `max()` тоже принимает аргумент `key`

In [17]:
categories = sorted(set(map(lambda p: p['Category'],
                            filter(lambda p: p['Category'] is not None, products))))
for cat in categories:
    best = max(filter(lambda p: p['Category'] == cat, products),
               key=lambda p: p['Price'])
    print(f"Категория {cat}: {best['Name']} - ${best['Price']:.2f} (ID {best['ID']})")

Категория A: Bouq All Italian - Primerba - $99.63 (ID 750)
Категория B: Country Roll - $99.76 (ID 664)
Категория C: Pork Loin Bine - In Frenched - $99.86 (ID 563)
Категория D: Juice - Grape, White - $99.63 (ID 155)


**4. Количество наименований товаров без скидки.** Считаем уникальные наименования (через множество), а товары с неуказанным признаком скидки не учитываем

In [18]:
no_discount = list(filter(lambda p: p['WithDiscount'] is False, products))
names_no_discount = set(map(lambda p: p['Name'], no_discount))
print('Записей без скидки:', len(no_discount))
print('Уникальных наименований без скидки:', len(names_no_discount))
print('Признак скидки не указан у', len(list(filter(lambda p: p['WithDiscount'] is None, products))), 'записей')

Записей без скидки: 372
Уникальных наименований без скидки: 347
Признак скидки не указан у 254 записей


**Вывод.** Лямбда-выражения позволили компактно задать ключи сортировки (в том числе по вычисляемому значению и по нескольким полям), условия фильтрации и правила преобразования записей. Комбинация `filter` → `map` → `sorted`/`max`/`sum` решает все четыре задачи варианта без явных циклов.

## Контрольные вопросы

**1. Что такое анонимная функция?**

Анонимная функция - функция без имени, объявляемая прямо в месте использования. В Python она создаётся выражением `lambda` и состоит из одного выражения, значение которого автоматически возвращается. Обычно её передают как аргумент в другие функции (`sorted`, `filter`, `map`, `max`).

**2. Опишите синтаксис объявления лямбда-выражения.**

```
lambda параметры: выражение
```
* после `lambda` через запятую перечисляются параметры (их может не быть; допустимы значения по умолчанию, `*args`, `**kwargs`);
* после двоеточия - **одно** выражение; `return` не пишется, результат выражения возвращается сам;
* операторы (`if` как инструкция, `for`, присваивание `=`) внутри лямбды недопустимы, но можно использовать условное выражение `a if условие else b`.

In [19]:
square = lambda x: x ** 2
add = lambda a, b=10: a + b
sign = lambda x: 'плюс' if x >= 0 else 'минус'
print(square(5), add(1), add(1, 2), sign(-3))

25 11 3 минус


**3. Поясните назначение лямбда-выражений. Приведите примеры их использования.**

Лямбды нужны, когда небольшая функция требуется один раз - как аргумент другой функции. Это избавляет от объявления отдельной функции через `def`. Типичные случаи: ключ сортировки (`sorted`, `list.sort`, `min`, `max`), условие фильтрации (`filter`), преобразование (`map`), обработчики событий в GUI. Если логика сложная, лучше обычная функция с именем: её проще читать и тестировать.

In [20]:
words = ['python', 'Arch', 'linux', 'Kali', 'nmap']
print(sorted(words, key=lambda w: w.lower()))
print(max(words, key=lambda w: len(w)))
print(list(map(lambda w: w.upper(), words)))

['Arch', 'Kali', 'linux', 'nmap', 'python']
python
['PYTHON', 'ARCH', 'LINUX', 'KALI', 'NMAP']


**4. Приведите пример использования лямбда-выражений для фильтрации элементов списка.**

In [21]:
numbers = [12, -5, 7, 0, 33, -18, 4, 9]
even = list(filter(lambda x: x % 2 == 0, numbers))
positive_odd = list(filter(lambda x: x > 0 and x % 2 == 1, numbers))
print('Чётные:', even)
print('Положительные нечётные:', positive_odd)

print([p['Name'] for p in filter(lambda p: p['Category'] == 'D' and p['Price'] > 98, products)])

Чётные: [12, 0, -18, 4]
Положительные нечётные: [7, 33, 9]
['Wine - Stoneliegh Sauvignon', 'Juice - Grape, White']


**5. Что будет напечатано в результате выполнения кода?**

Строка разбивается по пробелам на `['1.1', '3.3', '6.6']`, `map(float, ...)` превращает каждую подстроку в число, `x[1]` - второй элемент. Будет напечатано **`3.3`**.

In [22]:
sx = '1.1 3.3 6.6'
x = list(map(float, sx.split()))
print(x[1])

3.3


**6. Что будет напечатано в результате выполнения кода?**

`filter` оставляет только элементы, тип которых строго `int`. `1.3` - `float`, `'abs'` - `str`, они отбрасываются. Будет напечатано **`[13, 4, 78]`**.

In [23]:
data = [13, 1.3, 4, 'abs', 78]
filtered_data = list(filter(lambda x: type(x) == int, data))
print(filtered_data)

[13, 4, 78]
